In [10]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn
from sklearn.preprocessing import StandardScaler


import numpy as np
import polars as pl
import pandas as pd
import gsd.hoomd
import schmeud
import schmeud._schmeud as schmeud_rs
from schmeud._schmeud import statics
from schmeud import ml
from tqdm import tqdm

import glob
import os
import pathlib
import pickle
import signac
import freud
from numba import njit

from dataclasses import dataclass
from collections import defaultdict

import matplotlib.pyplot as plt
from scipy import stats
from typing import Union

from dataset import GlassDataset

In [2]:
from monk import workflow, utils

In [3]:
cwd = pathlib.Path(os.getcwd())
parent = cwd.parent.parent.parent / "config.yaml"
config = workflow.get_config(parent.as_posix())
local = cwd / "config.yaml"
local_config = workflow.get_any_yaml(local.as_posix())
parent, config, local_config

(PosixPath('/home/ian/Projects/work/monk/workflows/2d-osc-shear/config.yaml'),
 {'root': '/media/ian/Data2/monk/2d-osc-shear',
  'origin': '/media/ian/Data2/monk/2d-esl'},
 {'data': '/media/ian/Poppy/improved-nn/d2min-v1'})

In [4]:
project: signac.Project = signac.get_project(root=config['root'])
project.doc

{'avail_seed': 18, 'dt': 0.005, 'step_unit': 200, 'equil_time': 100, 'min_periods': 20, 'dumps': 40, 'period_times': [30.0, 100.0, 300.0, 1000.0], 'max_shears': [0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.04, 0.06, 0.07], '_status': {}}

# Prepare dataset and save to local SSD

We already have softness and D2min computed for these systems. Loop over the
systems and save the data to local SSD.

## Simulations

I ran both thermal and athermal simulations of the KA mixture. Lets see what
datapoints I did for each

- experiments: short thermal simulations with varying $T$ and period $tau$
- longer_experiments: longer thermal simulations with varying $T$ at a single
  period
- aqs_therm_experiments: not really AQS, just low temperature and longer
- aqs_experiments: true AQS simulation, for 20 cycles
- aqs_experiments_preshear: true AQS simulation, for 20 cycles, with strong
  preshear applied at the start of the simulation
- aqs_experiments_longer: true AQS simulation, for 100 cycles